# LG HelloDoctor — B팀 의료 LLM 파인튜닝
> 데이터 로드 → LoRA 파인튜닝 → 의도분류 → Entity추출 → Groq fallback → 평가

## Step 1 — 라이브러리 설치

In [ ]:
!pip install unsloth trl datasets bitsandbytes groq -q
print('설치 완료!')

설치 완료!


## Step 2 — Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/model', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf', exist_ok=True)
print('Drive 연결 완료!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive 연결 완료!


## Step 3 — 데이터 로드

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files='/content/drive/MyDrive/LG_HelloDoctor/LLM/data/00_all_medical_train.jsonl',
    split='train'
)

print(f'데이터 로드 완료: {len(dataset)}개')
print('샘플 확인:')
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

데이터 로드 완료: 2500개
샘플 확인:
{'instruction': '있잖아요, 근데 저기요, 기침이 심해요', 'input': '', 'output': '많이 힘드시겠어요. 내과에 가보시겠어요?'}


## Step 4 — 모델 로드 (4bit 양자화)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/llama-3.2-3b-instruct',
    max_seq_length=512,
    load_in_4bit=True,
)
print('모델 로드 완료!')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


모델 로드 완료!


## Step 5 — LoRA 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing=True,
)
print('LoRA 설정 완료!')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA 설정 완료!


## Step 6 — 프롬프트 포맷

In [ ]:
def format_prompt(example):
    return {
        'text': f"""### 질문:\n{example['instruction']}\n\n### 답변:\n{example['output']}<|end_of_text|>"""
    }

dataset = dataset.map(format_prompt)
print('프롬프트 포맷 완료!')
print('샘플:', dataset[0]['text'][:100])

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

프롬프트 포맷 완료!
샘플: ### 질문:
있잖아요, 근데 저기요, 기침이 심해요

### 답변:
많이 힘드시겠어요. 내과에 가보시겠어요?<|end_of_text|>


## Step 7 — 학습

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    args=TrainingArguments(
        output_dir='/content/drive/MyDrive/LG_HelloDoctor/LLM/checkpoints',
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        save_total_limit=3,
        resume_from_checkpoint=True,
    ),
)

trainer.train()
print('학습 완료!')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,500 | Num Epochs = 3 | Total steps = 471
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,2.666035
20,1.428617
30,0.953109
40,0.703708
50,0.562568
60,0.488914
70,0.458924
80,0.394291
90,0.409127
100,0.357646


학습 완료!


## Step 8 — 모델 저장

In [ ]:
model.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor/LLM/model')
tokenizer.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor/LLM/model')
print('모델 저장 완료!')

모델 저장 완료!


## Step 9 — GGUF 변환

In [ ]:
model.save_pretrained_gguf(
    '/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('GGUF 변환 완료!')

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:01<00:00, 60.69s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:45<00:00, 112.58s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/Modelfile
GGUF 변환 완료!


In [1]:
# 1. 설치
!pip install unsloth groq -q

# 2. Drive 연결
from google.colab import drive
drive.mount('/content/drive')

# 3. 저장된 모델 불러오기
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='/content/drive/MyDrive/LG_HelloDoctor/LLM/model',
    max_seq_length=512,
    load_in_4bit=True,
)
print('모델 불러오기 완료!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2

model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


모델 불러오기 완료!


## Step 10 — 의도 분류기

In [2]:
import re

# 응급 키워드 (LLM 거치지 않고 즉시 판단)
EMERGENCY_KEYWORDS = [
    '숨이 안 쉬어', '가슴이 너무 아프', '의식이 없',
    '쓰러', '피를 토', '말이 안 나', '한쪽이 마비',
    '갑자기 안 보여', '갑자기 못 움직', '말이 어눌',
    '입이 돌아', '숨을 못 쉬', '119'
]

INTENT_PROMPT = """다음 문장의 의도를 아래 4가지 중 하나로만 답하세요.
의도 종류:
- symptom_inquiry: 증상 문의 또는 진료과 질문
- hospital_search: 병원 위치, 운영시간 문의
- medication_info: 약 복용, 약 정보 문의
- emergency: 응급 상황

문장: {text}
의도:"""

def classify_intent(text: str) -> dict:
    # 1순위: 응급 키워드 즉시 판단
    for kw in EMERGENCY_KEYWORDS:
        if kw in text:
            return {'intent': 'emergency', 'confidence': 0.99, 'method': 'keyword'}

    # 2순위: 파인튜닝 모델로 분류
    prompt = INTENT_PROMPT.format(text=text)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=20,
        temperature=0.1,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split('의도:')[-1].strip().split()[0]

    valid_intents = ['symptom_inquiry', 'hospital_search', 'medication_info', 'emergency']
    intent = answer if answer in valid_intents else 'symptom_inquiry'

    return {'intent': intent, 'confidence': 0.85, 'method': 'llm'}


# 테스트
test_cases = [
    '무릎이 너무 아파요',
    '가슴이 너무 아프고 숨이 안 쉬어져요',
    '혈압약이랑 감기약 같이 먹어도 되나요?',
    '가까운 내과 어디예요?',
]

print('=== 의도 분류 테스트 ===')
for text in test_cases:
    result = classify_intent(text)
    print(f'입력: {text}')
    print(f'결과: {result}')
    print('-' * 40)

=== 의도 분류 테스트 ===


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

입력: 무릎이 너무 아파요
결과: {'intent': 'symptom_inquiry', 'confidence': 0.85, 'method': 'llm'}
----------------------------------------
입력: 가슴이 너무 아프고 숨이 안 쉬어져요
결과: {'intent': 'emergency', 'confidence': 0.99, 'method': 'keyword'}
----------------------------------------


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 혈압약이랑 감기약 같이 먹어도 되나요?
결과: {'intent': 'symptom_inquiry', 'confidence': 0.85, 'method': 'llm'}
----------------------------------------
입력: 가까운 내과 어디예요?
결과: {'intent': 'symptom_inquiry', 'confidence': 0.85, 'method': 'llm'}
----------------------------------------


## Step 11 — Entity 추출 (증상·부위·위치)

In [3]:
import json

BODY_PARTS = [
    '무릎', '허리', '어깨', '팔', '다리', '발', '손', '목',
    '머리', '눈', '귀', '코', '입', '치아', '잇몸', '가슴',
    '배', '위', '심장', '폐', '피부', '발목', '손목', '골반'
]

ENTITY_PROMPT = """다음 문장에서 증상과 신체 부위를 추출해서 JSON으로만 답하세요.
형식: {{"symptom": "증상", "body_part": "신체부위", "location": null}}
없으면 null로 표시하세요.

문장: {text}
JSON:"""

def extract_entities(text: str) -> dict:
    # 키워드 기반 빠른 추출
    found_parts = [p for p in BODY_PARTS if p in text]
    body_part = found_parts[0] if found_parts else None

    # LLM으로 정확한 추출
    prompt = ENTITY_PROMPT.format(text=text)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=60,
        temperature=0.1,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_str = result.split('JSON:')[-1].strip()

    try:
        json_match = re.search(r'\{.*?\}', json_str, re.DOTALL)
        entities = json.loads(json_match.group()) if json_match else None
    except:
        entities = None

    if not entities:
        entities = {'symptom': None, 'body_part': body_part, 'location': None}

    return entities


# 테스트
test_cases = [
    '무릎이 너무 아파요. 어디 가야 해요?',
    '허리가 끊어질 것 같아요',
    '혈압약이랑 감기약 같이 먹어도 되나요?',
]

print('=== Entity 추출 테스트 ===')
for text in test_cases:
    entities = extract_entities(text)
    print(f'입력: {text}')
    print(f'결과: {entities}')
    print('-' * 40)

Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Entity 추출 테스트 ===


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 무릎이 너무 아파요. 어디 가야 해요?
결과: {'symptom': '무릎이 너무 아파요', 'body_part': '무릎', 'location': None}
----------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 허리가 끊어질 것 같아요
결과: {'symptom': '허리가 끊어질 것 같아요', 'body_part': '허리', 'location': None}
----------------------------------------
입력: 혈압약이랑 감기약 같이 먹어도 되나요?
결과: {'symptom': None, 'body_part': None, 'location': None}
----------------------------------------


## Step 12 — Groq Fallback 연동

In [6]:
from groq import Groq
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

groq_client = Groq(api_key=GROQ_API_KEY)


SYSTEM_PROMPT = """당신은 노인 환자를 위한 의료 안내 AI 헬로비입니다.
반드시 아래 규칙을 지키세요:
- 3문장 이내로 답하세요
- 쉬운 말로 부드럽게 답하세요
- 존댓말을 사용하세요
- 질환을 단정하지 마세요
- 응급 상황이면 119를 먼저 안내하세요
- 금지 단어: 예후, 처방전, 투약, 병변"""

def generate_answer_local(text: str, context: str = '') -> dict:
    """파인튜닝 모델로 답변 생성"""
    prompt = f'### 질문:\n{text}\n\n### 답변:\n'
    if context:
        prompt = f'참고 정보: {context}\n\n{prompt}'

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split('### 답변:')[-1].strip()
    return {'answer': answer, 'model': 'LG_HelloDoctor/LLM'}


def generate_answer_groq(text: str, context: str = '') -> dict:
    """Groq API로 답변 생성 (fallback)"""
    user_msg = f'참고 정보: {context}\n\n질문: {text}' if context else text

    response = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg}
        ],
        max_tokens=150,
        temperature=0.3,
    )
    answer = response.choices[0].message.content.strip()
    return {'answer': answer, 'model': 'groq-llama-3.3-70b'}


def generate_answer(text: str, context: str = '', confidence: float = 0.85) -> dict:
    """신뢰도 기반 라우팅"""
    if confidence >= 0.7:
        try:
            return generate_answer_local(text, context)
        except Exception as e:
            print(f'로컬 오류 → Groq fallback: {e}')
            return generate_answer_groq(text, context)
    else:
        print('신뢰도 낮음 → Groq fallback')
        return generate_answer_groq(text, context)


# 테스트
print('=== 로컬 모델 ===')
r = generate_answer('무릎이 너무 아파요. 어디 가야 해요?', confidence=0.91)
print(f'모델: {r["model"]}\n답변: {r["answer"]}')

print('\n=== Groq Fallback ===')
r = generate_answer('무릎이 너무 아파요. 어디 가야 해요?', confidence=0.5)
print(f'모델: {r["model"]}\n답변: {r["answer"]}')

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 로컬 모델 ===


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


모델: LG_HelloDoctor/LLM
답변: 많이 아프시겠어요. 정형외과에 가보시는 게 좋을 것 같아요.

=== Groq Fallback ===
신뢰도 낮음 → Groq fallback
모델: groq-llama-3.3-70b
답변: 무릎이 아프면 병원에 가서 의사 선생님께 진찰을 받는 것이 좋을 것 같습니다. 근처에 정형외과나 내과가 있는 병원을 찾아보시는 것이 좋을 것 같아요. 혹시 심하게 아파서 바로 도와야 한다면 119에 연락하세요.


## Step 13 — 모델 평가 및 성능 비교

In [7]:
TEST_SET = [
    ('발목이 삐끗했어요', 'symptom_inquiry', '정형외과'),
    ('눈이 너무 빨개요', 'symptom_inquiry', '안과'),
    ('소화제 먹어도 돼요?', 'medication_info', None),
    ('갑자기 말이 안 나와요', 'emergency', None),
    ('가까운 내과 알려주세요', 'hospital_search', None),
    ('혈압약 언제 먹어요?', 'medication_info', None),
    ('어깨가 너무 아파서 팔을 못 들겠어요', 'symptom_inquiry', '정형외과'),
    ('기침이 한 달째 안 멈춰요', 'symptom_inquiry', '내과'),
    ('가슴이 쥐어짜는 것 같아요', 'emergency', None),
    ('피부에 두드러기가 났어요', 'symptom_inquiry', '피부과'),
    ('당뇨약이랑 두통약 같이 먹어도 돼요?', 'medication_info', None),
    ('귀에서 삐 소리가 나요', 'symptom_inquiry', '이비인후과'),
    ('쓰러졌어요', 'emergency', None),
    ('치아가 흔들려요', 'symptom_inquiry', '치과'),
    ('소변 볼 때 따가워요', 'symptom_inquiry', '비뇨의학과'),
    ('약을 깜빡했는데 지금 먹어도 돼요?', 'medication_info', None),
    ('머리가 갑자기 너무 아파요', 'symptom_inquiry', '신경과'),
    ('숨쉬기가 힘들어요', 'emergency', None),
    ('배에서 소리가 나고 설사를 해요', 'symptom_inquiry', '내과'),
    ('손이 자꾸 저려요', 'symptom_inquiry', '정형외과'),
]

def evaluate_model():
    intent_correct = 0
    dept_correct = 0
    dept_total = 0
    emergency_correct = 0
    emergency_total = 0
    korean_correct = 0
    wrong_cases = []

    for text, true_intent, true_dept in TEST_SET:
        intent_result = classify_intent(text)
        pred_intent = intent_result['intent']
        answer_result = generate_answer(text, confidence=intent_result['confidence'])
        answer = answer_result['answer']

        intent_ok = pred_intent == true_intent
        if intent_ok:
            intent_correct += 1
        else:
            wrong_cases.append({'text': text, 'true': true_intent, 'pred': pred_intent})

        if true_intent == 'emergency':
            emergency_total += 1
            if pred_intent == 'emergency':
                emergency_correct += 1

        if true_dept:
            dept_total += 1
            if true_dept in answer:
                dept_correct += 1

        korean_ratio = sum(1 for c in answer if '가' <= c <= '힣') / max(len(answer), 1)
        if korean_ratio > 0.3:
            korean_correct += 1

    total = len(TEST_SET)
    print('=' * 50)
    print('📊 모델 평가 결과')
    print('=' * 50)
    print(f'의도 분류 정확도:  {intent_correct}/{total} = {intent_correct/total*100:.1f}%  (목표: 80%↑)')
    print(f'응급 감지율:       {emergency_correct}/{emergency_total} = {emergency_correct/emergency_total*100:.1f}%  (목표: 95%↑)')
    print(f'진료과 정확도:     {dept_correct}/{dept_total} = {dept_correct/dept_total*100:.1f}%  (목표: 80%↑)')
    print(f'한국어 응답률:     {korean_correct}/{total} = {korean_correct/total*100:.1f}%  (목표: 99%↑)')
    print('=' * 50)

    if wrong_cases:
        print('\n❌ 틀린 의도 분류:')
        for w in wrong_cases:
            print(f'  입력: {w["text"]}')
            print(f'  정답: {w["true"]} / 예측: {w["pred"]}')

evaluate_model()

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_ge

📊 모델 평가 결과
의도 분류 정확도:  14/20 = 70.0%  (목표: 80%↑)
응급 감지율:       3/4 = 75.0%  (목표: 95%↑)
진료과 정확도:     10/11 = 90.9%  (목표: 80%↑)
한국어 응답률:     20/20 = 100.0%  (목표: 99%↑)

❌ 틀린 의도 분류:
  입력: 소화제 먹어도 돼요?
  정답: medication_info / 예측: symptom_inquiry
  입력: 가까운 내과 알려주세요
  정답: hospital_search / 예측: medication_info
  입력: 혈압약 언제 먹어요?
  정답: medication_info / 예측: symptom_inquiry
  입력: 당뇨약이랑 두통약 같이 먹어도 돼요?
  정답: medication_info / 예측: symptom_inquiry
  입력: 머리가 갑자기 너무 아파요
  정답: symptom_inquiry / 예측: emergency
  입력: 숨쉬기가 힘들어요
  정답: emergency / 예측: symptom_inquiry


## Step 14 — A팀 출력 받아서 전체 파이프라인 통합 테스트

In [8]:
import json

def full_pipeline(input_from_A: dict, rag_context: str = '') -> dict:
    """
    A팀 출력을 받아서 처리 후 C팀·D팀에 전달

    input_from_A 형식:
    {
        'text':       '무릎 통증. 진료 병원 문의',
        'raw_text':   '무릎이 너무 아파요. 어디 가야 하나요?',
        'confidence': 0.94,
        'language':   'ko'
    }
    """
    text = input_from_A['text']
    print(f'입력 텍스트: {text}')
    print('-' * 40)

    # 1. 의도 분류
    intent_result = classify_intent(text)
    print(f'① 의도: {intent_result["intent"]} (신뢰도: {intent_result["confidence"]})')

    # 2. Entity 추출
    entities = extract_entities(text)
    print(f'② Entity: {entities}')

    # 3. 응급이면 즉시 응답
    if intent_result['intent'] == 'emergency':
        answer = '지금 바로 119에 전화해 주세요. 매우 위험한 상황이에요.'
        print(f'③ 응급 즉시 응답')
        output = {
            'intent': 'emergency',
            'entities': entities,
            'answer': answer,
            'model': 'emergency_handler'
        }
        print(f'\nC팀·D팀 전달 JSON:')
        print(json.dumps(output, ensure_ascii=False, indent=2))
        return output

    # 4. 답변 생성
    answer_result = generate_answer(
        text,
        context=rag_context,
        confidence=intent_result['confidence']
    )
    print(f'③ 사용 모델: {answer_result["model"]}')

    # 5. C팀·D팀 전달 형식
    output = {
        'intent':   intent_result['intent'],
        'entities': entities,
        'answer':   answer_result['answer'],
        'model':    answer_result['model']
    }

    print(f'\nC팀·D팀 전달 JSON:')
    print(json.dumps(output, ensure_ascii=False, indent=2))
    return output


# 시나리오 A — 증상 문의
print('=== 시나리오 A ===')
full_pipeline({
    'text': '무릎 통증. 진료 병원 문의',
    'raw_text': '무릎이 너무 아파요. 어디 가야 하나요?',
    'confidence': 0.94,
    'language': 'ko'
})

print()

# 시나리오 B — 응급
print('=== 시나리오 B ===')
full_pipeline({
    'text': '가슴이 너무 아프고 숨이 안 쉬어져요',
    'raw_text': '헬로비야 가슴이 너무 아프고 숨이 안 쉬어져요',
    'confidence': 0.97,
    'language': 'ko'
})

print()

# 시나리오 C — 복약
print('=== 시나리오 C ===')
full_pipeline({
    'text': '혈압약이랑 감기약 같이 먹어도 되나요',
    'raw_text': '헬로비 혈압약이랑 감기약 같이 먹어도 되나요',
    'confidence': 0.91,
    'language': 'ko'
})

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 시나리오 A ===
입력 텍스트: 무릎 통증. 진료 병원 문의
----------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


① 의도: hospital_search (신뢰도: 0.85)


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


① 의도: medication_info (신뢰도: 0.85)


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


② Entity: {'symptom': None, 'body_part': None, 'location': None}
③ 사용 모델: LG_HelloDoctor/LLM

C팀·D팀 전달 JSON:
{
  "intent": "medication_info",
  "entities": {
    "symptom": null,
    "body_part": null,
    "location": null
  },
  "answer": "두 약을 함께 드시면 안 될 수 있어요. 약사 선생님께 한번 여쭤봐 주시겠어요?",
  "model": "LG_HelloDoctor/LLM"
}


{'intent': 'medication_info',
 'entities': {'symptom': None, 'body_part': None, 'location': None},
 'answer': '두 약을 함께 드시면 안 될 수 있어요. 약사 선생님께 한번 여쭤봐 주시겠어요?',
 'model': 'LG_HelloDoctor/LLM'}

깃헙 토큰 발급

In [12]:
# 기존 폴더 삭제 후 다시 시작
import os
os.chdir('/content')
!rm -rf LGHelloDoctor



In [20]:
import os
os.chdir('/content')
!rm -rf LGHelloDoctor

!git clone https://YOUR_TOKEN_HERE@github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git
os.chdir('/content/LGHelloDoctor')
!git checkout llm
!cp /content/drive/MyDrive/LG_HelloDoctor/LLM/B_llm_full.ipynb .
!mkdir -p data
!cp /content/drive/MyDrive/LG_HelloDoctor/LLM/data/00_all_medical_train.jsonl data/
!git config user.email "dkswndus6988@naver.com"
!git config user.name "dkswndus"
!git add .
!git commit -m "feat: 의료 LLM 파인튜닝 코드 및 학습 데이터 추가"
!git push origin llm

Cloning into 'LGHelloDoctor'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 58 (delta 17), reused 45 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 209.70 KiB | 5.11 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Branch 'llm' set up to track remote branch 'llm' from 'origin'.
Switched to a new branch 'llm'
[llm c4822d5] feat: 의료 LLM 파인튜닝 코드 및 학습 데이터 추가
 2 files changed, 2501 insertions(+)
 create mode 100644 B_llm_full.ipynb
 create mode 100644 data/00_all_medical_train.jsonl
remote: Permission to lg-hellovision-dx-data-school/LGHelloDoctor.git denied to dkswndus.
fatal: unable to access 'https://github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git/': The requested URL returned error: 403
